# Wavelet Data Access (Load or Create)

This notebook prepares wavelet-transformed EEG data for direct use in your own analysis code.

- Set `REUSE_WAVELETS=True` to load previously stored wavelets when available.
- Set `REUSE_WAVELETS=False` to compute wavelets and store them for future reuse.

This notebook intentionally skips ISC/mean-variance analysis and plotting.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from scripts.analysis_common import (
    load_analyzers,
    analyzers_to_datasets,
    _wavelet_transform,
)
from src.definitions.fields import (
    ConditionVariants,
    MusicTypeVariants,
    ExclusionCategories,
)
from src.definitions.constants import ProjectPaths, ExperimentNames

## Configuration

In [ ]:
# Input selection
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL, MusicTypeVariants.PSYTRANCE]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# Data preparation
PROCESS_AND_SAVE_DATA = False

# Wavelet options
REPRESENTATION = "power"  # "power" or "phase"
FREQS = np.linspace(1, 40, 40)
KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_items, n_channels, n_freqs, n_samples)

# Reuse/load options
REUSE_WAVELETS = True
WAVELET_DIR = (
    ProjectPaths.PROCESSED_DATA_DIR
    / ExperimentNames.PSILO_MUSIC.value
    / "wavelets"
    / "notebook"
)
WAVELET_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet directory: {WAVELET_DIR}")

## Load source data

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
)
datasets = analyzers_to_datasets(analyzers)

print("Loaded datasets:", list(datasets.keys()))

## Load or compute wavelets

When `REUSE_WAVELETS=True`, existing wavelet files are loaded from `WAVELET_DIR`.
If a file is not present, wavelets are computed and saved.

In [ ]:
wavelet_datasets = _wavelet_transform(
    datasets=datasets,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=Path(WAVELET_DIR),
    reuse_wavelets=REUSE_WAVELETS,
)

for label, ad in wavelet_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"{label}: shape={ad.data.shape}, source={source}")

## Work with wavelet values directly

Use `wavelet_arrays[label]` as a NumPy array in downstream custom code.

In [ ]:
wavelet_arrays = {label: ad.data for label, ad in wavelet_datasets.items()}

# Example: pick first dataset and inspect dimensions
first_label = next(iter(wavelet_arrays))
x = wavelet_arrays[first_label]
print("First dataset:", first_label)
print("Array shape:", x.shape)
print("Array dtype:", x.dtype)